# Ch.1 演習ノートブック：データ読み込み・集計

| 項目 | 内容 |
|------|------|
| **使用データ** | ワインデータ（`load_wine()`）178件・13特徴量・3品種 |
| **作るもの** | 品種別・主要指標の集計表 |
| **実務での対応物** | 月次売上レポート・KPIダッシュボードの集計表 |
| **時間の目安** | 座学後の演習 40分（問3まで完了で十分） |

## このノートブックの使い方

各問題の前に **解説セル** があります。必ず読んでから取り組んでください。

**詰まったときの対処順番：**
1. 解説セルをもう一度読む
2. スライド「08｜コードで確認する」を見返す
3. Copilot に「〇〇するPythonコードを教えて」と聞く

> ✅ **問3まで完了すれば十分です。問4は時間が余ったときに挑戦してください。**

---
## 🤖 Copilot Chat の使い方

### 開き方：ウェブブラウザで開く

```
Microsoft Copilot → https://copilot.microsoft.com
GitHub Copilot   → https://github.com/copilot
```

> 画面左半分に JupyterLab、右半分にブラウザ（Copilot Chat）を並べて使ってください。  
> JupyterLab のサイドバー機能は使いません。

### 演習のコードはすべて Copilot Chat に書いてもらう

このコースでは **演習コードはすべて Copilot Chat に生成してもらうことを推奨** します。  
自分の仕事は「**何をしたいかを言語化すること**」と「**結果を解釈すること**」です。

### 質問の型（必ずこの形式でコピペして使う）

```
【やりたいこと】〇〇したい
【使うライブラリ】pandas, matplotlib など
【データの形】DataFrame で列名は〇〇、行数は〇〇件
【環境】Python 3.8.6、Windows、JupyterLab
【わからない点】〇〇の書き方 / 〇〇でエラーが出た
```

### Copilot 活用ルール

| タイミング | ルール |
|-----------|--------|
| 座学中 | **禁止** ― まず概念を頭に入れる |
| 演習 問1〜2 | **上の型でプロンプトを書いて質問する** |
| 演習 問3（考察） | コードはOK・**考察文は自分で書く** |
| 問4（発展） | **自由に活用** |

> 詳細テンプレート集 → `07_copilot_prompt_guide.md`

---
## 🔧 準備 ― ライブラリの読み込み

### なぜ import するのか
Python は「必要な道具箱だけを呼び出す」設計になっています。  
最初にこのセルを実行しておくことで、以降のセルで道具が使えるようになります。

| ライブラリ | 役割 |
|-----------|------|
| `pandas` | 表形式データの操作（Excelのような操作をコードで） |
| `numpy` | 数値計算の基盤（平均・標準偏差など） |
| `sklearn.datasets` | 練習用データセットの提供 |

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine

print("✅ ライブラリの読み込み完了")

### データを読み込んで DataFrame に変換する

`load_wine()` は scikit-learn が提供するワインデータです。  
178件・13特徴量・3品種（Barolo / Grignolino / Barbera）が含まれています。

**実務での対応：** 売上 CSV / 顧客データベース → `pd.read_csv("data.csv")` と同じ操作です。

In [ ]:
wine = load_wine()
df   = pd.DataFrame(wine.data, columns=wine.feature_names)
df["品種"] = wine.target   # 品種番号（0=Barolo, 1=Grignolino, 2=Barbera）

# 品種名の辞書でラベルをつける
species = {0: "Barolo", 1: "Grignolino", 2: "Barbera"}
df["品種名"] = df["品種"].map(species)

print(f"✅ データ読み込み完了！  行数: {len(df)}件  列数: {len(df.columns)}列")

---
## 📋 データの「形」を確認する（概念①）

> **実務での習慣：** 新しいデータをもらったら、分析を始める前に必ずこの3つを実行します。

### STEP 1：先頭5行を目で確認する（`head()`）

どんな列があるか・値の大きさの感覚を掴むための「最初の眺め」です。

In [ ]:
df.head()

### STEP 2：型・件数・欠損値を確認する（`info()`）

確認ポイント：
- `Non-Null Count` が行数（178）と一致している → 欠損なし
- `Dtype` が `float64` / `int64` → 数値として扱える

In [ ]:
df.info()

### STEP 3：基本統計量を確認する（`describe()`）

`std`（標準偏差）が大きい列 → 値のばらつきが大きい → 品種の差が出やすい変数の候補です。

In [ ]:
df.describe().round(2)

### STEP 4：欠損値を確認する（`isnull().sum()`）

**すべて 0 なら欠損なし。** 実務では欠損がある場合の方が多いです。

In [ ]:
missing = df.isnull().sum()
print(missing)
print(f"\n欠損値の合計: {missing.sum()} 件  ← 0 なら問題なし")

> 🤖 **Copilot Chat プロンプト**（右サイドバーのチャット欄にコピーして使ってください）
>  （詳細テンプレート：`07_copilot_prompt_guide.md` の [P1-1]）

> ```
> 【やりたいこと】pandasで欠損値がある列を平均値で補完したい
> 【使うライブラリ】pandas
> 【データの形】DataFrame で各列の欠損件数を isnull().sum() で確認済み
> 【環境】Python 3.8.6、Windows、JupyterLab
> 【わからない点】fillna(mean()) の使い方と、列ごとに適用する方法
> ```

---
## 問1：品種ごとのデータ件数を確認する ★☆☆

**実務での対応：** 「各商品カテゴリが何件あるか」「地域ごとの顧客数」を確認する作業と同じです。

### なぜやるのか
集計や比較をする前に「各グループが何件あるか」を確認するのが鉄則です。  
件数が偏っていると（例：品種Aが170件・品種Bが8件）、  
平均値の比較に注意が必要になります。

### `value_counts()` の使い方
```python
df["列名"].value_counts()   # 値ごとの件数を降順で表示
```

In [ ]:
# 品種番号ごとの件数
df["品種"].value_counts()

### 品種名でも確認してみましょう

In [ ]:
# 品種名ごとの件数（番号より読みやすい）
df["品種名"].value_counts()

#### ✅ チェックポイント
- 3品種の件数が表示されましたか？
- 合計は 178 件になっていますか？
- 大きな偏りはありませんか？（最大と最小の差が 2 倍以上なら偏りあり）

> 🤖 **Copilot Chat プロンプト**（右サイドバーのチャット欄にコピーして使ってください）
>  （詳細テンプレート：`07_copilot_prompt_guide.md` の [P1-1]）

> ```
> 【やりたいこと】pandasのDataFrameで品種列の値ごとに件数を集計して表示したい
> 【使うライブラリ】pandas
> 【データの形】DataFrame で「品種」列が 0/1/2 の整数、178件
> 【環境】Python 3.8.6、Windows、JupyterLab
> 【わからない点】value_counts() の使い方
> ```

---
## 問2：品種別の集計表を作る ★★☆

**実務での対応：** 「都道府県ごとの平均売上」「年代ごとの購入率」を集計するのと同じ操作です。

### `groupby()` の考え方：2ステップで覚える

```
ステップ1：df.groupby("品種名")         → 品種ごとにデータを仕分ける
ステップ2：.mean()                      → 各グループの平均を計算する
```

Excelの「ピボットテーブル」と同じことを、コード1行で実行できます。

### まず「アルコール度数」だけで試してみる

In [ ]:
# 品種ごとのアルコール度数の平均
df.groupby("品種名")["alcohol"].mean().round(2)

### 複数の列をまとめて集計する

In [ ]:
# 品種ごとにアルコール度数とプロリン含有量の平均を集計
result = df.groupby("品種名")[["alcohol", "proline"]].mean().round(2)
result

### 平均以外の集計（最大値・最小値・標準偏差）

In [ ]:
# 平均・最大・最小・標準偏差をまとめて確認
df.groupby("品種名")[["alcohol", "proline"]].agg(["mean", "max", "min", "std"]).round(2)

#### ✅ チェックポイント
- 3品種の平均値が表示されましたか？
- どの品種が最もアルコール度数・プロリンが高いですか？
- 標準偏差（std）が大きい品種はどれですか？（値のばらつきが大きい）

> 🤖 **Copilot Chat プロンプト**（右サイドバーのチャット欄にコピーして使ってください）
>  （詳細テンプレート：`07_copilot_prompt_guide.md` の [P1-2]）

> ```
> 【やりたいこと】pandasのgroupbyで品種ごとのアルコール度数を中央値・分位数でも集計したい
> 【使うライブラリ】pandas
> 【データの形】DataFrame で「品種名」列と「alcohol」「proline」列がある、178件
> 【環境】Python 3.8.6、Windows、JupyterLab
> 【わからない点】groupby().median() と groupby().quantile() の使い方
> ```

---
## 問3：集計結果から考察する ★★☆（最重要）

### なぜ考察が最重要なのか
**データサイエンティストの本来の仕事は「数字を読んで意味を伝えること」です。**

集計表を作るだけなら Excel でもできます。  
「この数字が何を意味するか」を言語化できる人が、実務で価値を発揮できます。

### 考察の書き方
```
「〇〇品種は、△△の値が他品種より□□倍高い/低い。
 これは□□という理由が考えられる。
 Ch.4 のモデルではこの変数が重要になると予想する。」
```

### 考察欄（ここに書いてください）

**気づいた品種・特徴量：**

> （例：品種〇〇は、プロリン含有量が他品種の約△倍になっている）

**なぜそうなると思いますか？**

> （正解なし。思ったことを自由に書いてください）

**Ch.4 のモデルで効くと思う変数は？**

> （理由も一言添えて記入してください）

---
## 問4（発展）：条件でデータを絞り込む ★★★

**実務での対応：** 「売上が高いかつ利益率も高い商品だけ抽出する」「特定地域の特定期間のデータだけ見る」と同じ操作です。

### 1条件フィルタ：アルコール度数が高いワインを絞り込む

Excelの「フィルター機能」と同じことをコードで実行できます。  
コードで書く利点：条件を変えて何度でも自動で実行できる。

In [ ]:
# アルコール度数が 13.5 以上のワインだけ取り出す
high_alc = df[df["alcohol"] >= 13.5]
print(f"条件を満たすワイン: {len(high_alc)} 件  （全体の {len(high_alc)/len(df):.0%}）")
high_alc[["品種名", "alcohol", "proline"]].head(10)

### 2条件フィルタ（AND 条件）

**重要：** 複数条件は `&` でつなぎ、各条件を `()` で囲む（括弧がないとエラー）

In [ ]:
# アルコール度数が 13.5 以上 かつ プロリンが 700 以上
premium = df[(df["alcohol"] >= 13.5) & (df["proline"] >= 700)]
print(f"両方の条件を満たすワイン: {len(premium)} 件")
premium["品種名"].value_counts()

### 絞り込んだデータでさらに集計する

In [ ]:
# 絞り込み後のデータで品種別平均を再計算
premium.groupby("品種名")[["alcohol", "proline"]].mean().round(2)

> 🤖 **Copilot Chat プロンプト**（右サイドバーのチャット欄にコピーして使ってください）
>  （詳細テンプレート：`07_copilot_prompt_guide.md` の [P1-3]）

> ```
> 【やりたいこと】アルコール度数が12.5以下 かつ proline が400以下のワインを取り出したい
> 【使うライブラリ】pandas
> 【データの形】DataFrame で「alcohol」「proline」列がある、178件
> 【環境】Python 3.8.6、Windows、JupyterLab
> 【わからない点】複数条件フィルタの書き方、& の使い方と括弧が必要な理由
> ```